# TESA 2025 Defensive Track — YOLO11-OBB Drone Detection Training

This notebook provides the complete end-to-end pipeline for fine-tuning **YOLO11-OBB (Oriented Bounding Boxes)** for aerial drone detection.

### Models Trained:
- **Train 6 (Nano Edge Model)**: `yolo11n-obb.pt` — 100 epochs, optimized for low-latency edge deployment on Raspberry Pi 5.
- **Train 10 (Champion Heavy Model)**: `yolo11x-obb.pt` — 200 epochs, maximum accuracy with rotated bounding boxes for tightly clustered aerial targets (97.7% mAP).

## 1. Environment Setup
Install `ultralytics` and dependencies, and check GPU availability.

In [ ]:
!pip install ultralytics sahi roboflow

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive or Load Dataset
Prepare dataset in YOLO-OBB format containing `data.yaml`.

In [ ]:
# If running on Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

# Or configure dataset path directly:
DATASET_YAML = "data.yaml"

## 3. Train YOLO11-OBB Models

### A. Train Edge Model (YOLO11n-OBB for Raspberry Pi 5)

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO11n-OBB backbone
nano_model = YOLO("yolo11n-obb.pt")

# Train for 100 epochs (Train 6 equivalent)
nano_results = nano_model.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu",
    project="runs/obb",
    name="yolo11n_drone_edge",
    plots=True,
    save=True
)

### B. Train Champion Model (YOLO11x-OBB for Maximum Precision)

In [ ]:
# Load pretrained YOLO11x-OBB backbone
heavy_model = YOLO("yolo11x-obb.pt")

# Train for 200 epochs (Train 10 equivalent)
heavy_results = heavy_model.train(
    data=DATASET_YAML,
    epochs=200,
    imgsz=640,
    batch=8,
    device=0 if torch.cuda.is_available() else "cpu",
    project="runs/obb",
    name="yolo11x_drone_champion",
    plots=True,
    save=True
)

## 4. Evaluation and Validation
Evaluate metrics (mAP50, mAP50-95, precision, recall) and plot confusion matrix.

In [ ]:
# Validate Nano model
metrics = nano_model.val()
print(f"Nano mAP50-95(B): {metrics.box.map:.4f}")
print(f"Nano mAP50: {metrics.box.map50:.4f}")

## 5. Export Model Weights
Export best weights to PyTorch `.pt`, ONNX, or NCNN for deployment.

In [ ]:
# Export best checkpoint to deployment models directory
import shutil
from pathlib import Path

best_pt = Path("runs/obb/yolo11n_drone_edge/weights/best.pt")
if best_pt.exists():
    target_path = Path("models/yolo11n-obb.pt")
    target_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(best_pt, target_path)
    print(f"Copied best weights to: {target_path}")

# Optional ONNX export for edge acceleration
# nano_model.export(format="onnx")